<a href="https://colab.research.google.com/github/CharalampiaKal/cicids2017-progressive-feature-reduction/blob/main/notebooks/01_data_audit_and_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Progressive Feature Reduction for Machine-Learning-Based Network Intrusion Detection

## Notebook 01: Data Audit and Cleaning

This notebook prepares the corrected CICIDS2017 dataset for the dissertation experiment. It downloads the improved dataset from the official DistriNet research source, verifies the five daily CSV files, audits their schemas and label distributions, and constructs the binary classification target while retaining the corrected attack labels for per-attack evaluation.

### Key methodological decisions

- Flows identified as attempted attacks are treated as BENIGN, following the recommendation of the corrected dataset's authors.
- The row ID, Flow ID, source and destination IP addresses, and timestamp are excluded to reduce identifier-based shortcut learning and data leakage.
- The raw dataset is downloaded during execution and is not stored in the GitHub repository.
- Source and destination ports and protocol are provisionally retained as behavioral network-flow features.


In [3]:
from pathlib import Path

# Create temporary dataset folders in the Colab runtime
DATA_DIR = Path("/content/data")
RAW_DIR = DATA_DIR / "raw"

DATA_DIR.mkdir(exist_ok=True)
RAW_DIR.mkdir(exist_ok=True)

# Official corrected CICIDS2017 dataset
DATASET_URL = (
    "https://intrusion-detection.distrinet-research.be/"
    "CNS2022/Datasets/CICIDS2017_improved.zip"
)
ZIP_PATH = DATA_DIR / "CICIDS2017_improved.zip"

# Download the dataset directly to Colab
!wget -c "$DATASET_URL" -O "$ZIP_PATH"

# Verify the download
print("\nDataset downloaded:", ZIP_PATH.exists())

if ZIP_PATH.exists():
    size_mb = ZIP_PATH.stat().st_size / (1024 ** 2)
    print(f"File size: {size_mb:.2f} MB")

--2026-08-23 18:16:30--  https://intrusion-detection.distrinet-research.be/CNS2022/Datasets/CICIDS2017_improved.zip
Resolving intrusion-detection.distrinet-research.be (intrusion-detection.distrinet-research.be)... 134.58.40.205
Connecting to intrusion-detection.distrinet-research.be (intrusion-detection.distrinet-research.be)|134.58.40.205|:443... connected.
HTTP request sent, awaiting response... 416 Requested Range Not Satisfiable

    The file is already fully retrieved; nothing to do.


Dataset downloaded: True
File size: 327.63 MB


In [4]:
import zipfile

# Extract the corrected dataset
with zipfile.ZipFile(ZIP_PATH, "r") as zip_file:
    zip_file.extractall(RAW_DIR)

# Find and display all extracted CSV files
csv_files = sorted(RAW_DIR.rglob("*.csv"))

print("CSV files extracted:", len(csv_files))

for file_path in csv_files:
    size_mb = file_path.stat().st_size / (1024 ** 2)
    print(f"{file_path.name}: {size_mb:.2f} MB")

CSV files extracted: 5
friday.csv: 271.98 MB
monday.csv: 198.25 MB
thursday.csv: 180.74 MB
tuesday.csv: 170.13 MB
wednesday.csv: 277.80 MB


In [5]:
import pandas as pd

# Read only the column names without loading the full dataset
schemas = {}

for file_path in csv_files:
    columns = pd.read_csv(file_path, nrows=0).columns.str.strip().tolist()
    schemas[file_path.name] = columns

# Use the first file as the reference schema
reference_file = csv_files[0].name
reference_columns = schemas[reference_file]

for filename, columns in schemas.items():
    print(
        f"{filename}: {len(columns)} columns | "
        f"Matches reference: {columns == reference_columns}"
    )

print("\nAll files have the same schema:",
      all(columns == reference_columns for columns in schemas.values()))

print("\nFirst 10 columns:")
print(reference_columns[:10])

print("\nLast 10 columns:")
print(reference_columns[-10:])

friday.csv: 91 columns | Matches reference: True
monday.csv: 91 columns | Matches reference: True
thursday.csv: 91 columns | Matches reference: True
tuesday.csv: 91 columns | Matches reference: True
wednesday.csv: 91 columns | Matches reference: True

All files have the same schema: True

First 10 columns:
['id', 'Flow ID', 'Src IP', 'Src Port', 'Dst IP', 'Dst Port', 'Protocol', 'Timestamp', 'Flow Duration', 'Total Fwd Packet']

Last 10 columns:
['Active Min', 'Idle Mean', 'Idle Std', 'Idle Max', 'Idle Min', 'ICMP Code', 'ICMP Type', 'Total TCP Flow Time', 'Label', 'Attempted Category']


In [7]:
from collections import Counter

file_audit = []
provided_label_counts = Counter()
modeling_label_counts = Counter()
attempted_category_counts = Counter()

# Process the dataset in chunks to avoid memory problems
for file_path in csv_files:
    file_rows = 0
    file_attempted = 0

    for chunk in pd.read_csv(
        file_path,
        usecols=["Label", "Attempted Category"],
        chunksize=200_000,
        low_memory=False
    ):
        labels = chunk["Label"].astype(str).str.strip()

        attempted_categories = pd.to_numeric(
            chunk["Attempted Category"],
            errors="coerce"
        ).fillna(-1).astype(int)

        is_attempted = attempted_categories.ne(-1)

        # Preserve the labels supplied in the dataset
        provided_label_counts.update(labels.value_counts().to_dict())

        # Following the dataset authors' recommendation,
        # attempted attacks will be treated as benign
        modeling_labels = labels.mask(is_attempted, "BENIGN")
        modeling_label_counts.update(modeling_labels.value_counts().to_dict())

        attempted_category_counts.update(
            attempted_categories[is_attempted].value_counts().to_dict()
        )

        file_rows += len(chunk)
        file_attempted += int(is_attempted.sum())

    file_audit.append({
        "File": file_path.name,
        "Rows": file_rows,
        "Attempted flows": file_attempted
    })

file_audit_df = pd.DataFrame(file_audit)

modeling_distribution_df = (
    pd.DataFrame(
        modeling_label_counts.items(),
        columns=["Label", "Count"]
    )
    .sort_values("Count", ascending=False)
    .reset_index(drop=True)
)

print("Total rows:", file_audit_df["Rows"].sum())
print("Total attempted flows:", file_audit_df["Attempted flows"].sum())

print("\nRows and attempted flows per file:")
display(file_audit_df)

print("\nClass distribution after attempted flows are treated as benign:")
display(modeling_distribution_df)

Total rows: 2099976
Total attempted flows: 11979

Rows and attempted flows per file:


,File,Rows,Attempted flows
0,friday.csv,547557,4067
1,monday.csv,371624,0
2,thursday.csv,362076,1997
3,tuesday.csv,322078,39
4,wednesday.csv,496641,5876



Class distribution after attempted flows are treated as benign:


,Label,Count
0,BENIGN,1594545
1,Portscan,159066
2,DoS Hulk,158468
3,DDoS,95144
4,Infiltration - Portscan,71767
5,DoS GoldenEye,7567
6,FTP-Patator,3972
7,DoS Slowloris,3859
8,SSH-Patator,2961
9,DoS Slowhttptest,1740


In [8]:
# Load a small sample from every file for structural inspection
sample_parts = []

for file_path in csv_files:
    sample = pd.read_csv(file_path, nrows=1_000, low_memory=False)
    sample.columns = sample.columns.str.strip()
    sample["source_file"] = file_path.name
    sample_parts.append(sample)

sample_df = pd.concat(sample_parts, ignore_index=True)

print("Sample shape:", sample_df.shape)

print("\nColumn types:")
print(sample_df.dtypes.value_counts())

non_numeric_columns = sample_df.select_dtypes(exclude="number").columns.tolist()

print("\nNon-numeric columns:")
print(non_numeric_columns)

print("\nSelected sample values:")
display(
    sample_df[
        ["Label", "Attempted Category", "source_file"]
    ].head(10)
)

Sample shape: (5000, 92)

Column types:
int64      61
float64    25
object      6
Name: count, dtype: int64

Non-numeric columns:
['Flow ID', 'Src IP', 'Dst IP', 'Timestamp', 'Label', 'source_file']

Selected sample values:


,Label,Attempted Category,source_file
0,BENIGN,-1,friday.csv
1,BENIGN,-1,friday.csv
2,BENIGN,-1,friday.csv
3,BENIGN,-1,friday.csv
4,BENIGN,-1,friday.csv
5,BENIGN,-1,friday.csv
6,BENIGN,-1,friday.csv
7,BENIGN,-1,friday.csv
8,BENIGN,-1,friday.csv
9,BENIGN,-1,friday.csv


In [9]:
import numpy as np

# Columns excluded from machine-learning features
identifier_columns = [
    "id",
    "Flow ID",
    "Src IP",
    "Dst IP",
    "Timestamp"
]

metadata_columns = ["Label", "Attempted Category"]

model_feature_columns = [
    column for column in reference_columns
    if column not in identifier_columns + metadata_columns
]

print("Identifier columns excluded:", identifier_columns)
print("Model features retained:", len(model_feature_columns))

data_parts = []

# Load the dataset in manageable chunks
for file_path in csv_files:
    for chunk in pd.read_csv(
        file_path,
        usecols=model_feature_columns + metadata_columns,
        chunksize=100_000,
        low_memory=False
    ):
        provided_labels = chunk["Label"].astype(str).str.strip()

        attempted_categories = pd.to_numeric(
            chunk["Attempted Category"],
            errors="coerce"
        ).fillna(-1).astype("int8")

        # Dataset authors recommend treating attempted attacks as benign
        diagnostic_labels = provided_labels.mask(
            attempted_categories.ne(-1),
            "BENIGN"
        )

        prepared_chunk = chunk[model_feature_columns].apply(
            pd.to_numeric,
            errors="coerce"
        )

        prepared_chunk["provided_label"] = provided_labels
        prepared_chunk["diagnostic_label"] = diagnostic_labels
        prepared_chunk["binary_target"] = (
            diagnostic_labels.ne("BENIGN").astype("int8")
        )
        prepared_chunk["attempted_category"] = attempted_categories
        prepared_chunk["source_file"] = file_path.name

        data_parts.append(prepared_chunk)

    print(f"Loaded: {file_path.name}")

full_df = pd.concat(data_parts, ignore_index=True)
del data_parts

# Reduce memory used by repeated text values
for column in ["provided_label", "diagnostic_label", "source_file"]:
    full_df[column] = full_df[column].astype("category")

memory_gb = full_df.memory_usage(deep=True).sum() / (1024 ** 3)

print("\nFull dataset shape:", full_df.shape)
print("Machine-learning features:", len(model_feature_columns))
print(f"Memory usage: {memory_gb:.2f} GB")

print("\nBinary target distribution:")
print(full_df["binary_target"].value_counts().sort_index())

Identifier columns excluded: ['id', 'Flow ID', 'Src IP', 'Dst IP', 'Timestamp']
Model features retained: 84
Loaded: friday.csv
Loaded: monday.csv
Loaded: thursday.csv
Loaded: tuesday.csv
Loaded: wednesday.csv

Full dataset shape: (2099976, 89)
Machine-learning features: 84
Memory usage: 1.32 GB

Binary target distribution:
binary_target
0    1594545
1     505431
Name: count, dtype: int64


## Data Quality Audit

This section examines the candidate model features for missing values, infinite values, and globally constant columns. The audit describes the complete corrected dataset. Any preprocessing parameters used for modelling will later be learned from the training partition only to prevent data leakage.

In [10]:
quality_records = []

for column in model_feature_columns:
    series = full_df[column]

    quality_records.append({
        "Feature": column,
        "Data type": str(series.dtype),
        "Missing values": int(series.isna().sum()),
        "Infinite values": int(np.isinf(series.to_numpy()).sum()),
        "Unique values": int(series.nunique(dropna=True))
    })

quality_audit_df = pd.DataFrame(quality_records)

quality_audit_df["Constant"] = (
    quality_audit_df["Unique values"] <= 1
)

problematic_quality_df = quality_audit_df[
    (quality_audit_df["Missing values"] > 0) |
    (quality_audit_df["Infinite values"] > 0) |
    (quality_audit_df["Constant"])
].reset_index(drop=True)

print("Total missing values:",
      quality_audit_df["Missing values"].sum())

print("Total infinite values:",
      quality_audit_df["Infinite values"].sum())

print("Globally constant features:",
      quality_audit_df["Constant"].sum())

print("\nFeatures requiring attention:")
display(problematic_quality_df)

Total missing values: 0
Total infinite values: 10
Globally constant features: 0

Features requiring attention:


,Feature,Data type,Missing values,Infinite values,Unique values,Constant
0,Flow Bytes/s,float64,0,5,1375061,False
1,Flow Packets/s,float64,0,5,1002315,False


## Duplicate-Flow Audit

Exact duplicates are identified using the 84 candidate model features together with the corrected diagnostic label. Identifier fields and the source filename are excluded from the duplicate definition. Duplicates are removed before partitioning to prevent identical observations from appearing in both training and evaluation data.

In [11]:
duplicate_subset = model_feature_columns + ["diagnostic_label"]

rows_before_duplicates = len(full_df)

# Keep the first occurrence of every identical feature-label combination
duplicate_mask = full_df.duplicated(
    subset=duplicate_subset,
    keep="first"
)

exact_duplicate_count = int(duplicate_mask.sum())
duplicate_percentage = (
    exact_duplicate_count / rows_before_duplicates * 100
)

duplicates_by_file_df = (
    full_df.loc[duplicate_mask]
    .groupby("source_file", observed=True)
    .size()
    .rename("Duplicates removed")
    .reset_index()
)

print("Rows before duplicate removal:", rows_before_duplicates)
print("Exact duplicates:", exact_duplicate_count)
print(f"Duplicate percentage: {duplicate_percentage:.2f}%")

print("\nDuplicates by source file:")
display(duplicates_by_file_df)

# Remove duplicates only from the in-memory working dataset
full_df = (
    full_df.loc[~duplicate_mask]
    .copy()
    .reset_index(drop=True)
)

del duplicate_mask

print("\nRows after duplicate removal:", len(full_df))

print("\nClass distribution after duplicate removal:")
post_duplicate_distribution_df = (
    full_df["diagnostic_label"]
    .value_counts()
    .rename_axis("Label")
    .reset_index(name="Count")
)

display(post_duplicate_distribution_df)

Rows before duplicate removal: 2099976
Exact duplicates: 15255
Duplicate percentage: 0.73%

Duplicates by source file:


,source_file,Duplicates removed
0,friday.csv,1549
1,monday.csv,2117
2,thursday.csv,5839
3,tuesday.csv,2716
4,wednesday.csv,3034



Rows after duplicate removal: 2084721

Class distribution after duplicate removal:


,Label,Count
0,BENIGN,1582444
1,Portscan,159059
2,DoS Hulk,158468
3,DDoS,95144
4,Infiltration - Portscan,68620
5,DoS GoldenEye,7567
6,FTP-Patator,3972
7,DoS Slowloris,3859
8,SSH-Patator,2961
9,DoS Slowhttptest,1740


## Stratified Training, Validation and Test Split

The deduplicated dataset is divided into 70% training, 15% validation, and 15% test partitions. Stratification uses the corrected diagnostic attack label rather than only the binary target. This helps preserve rare attack categories across the partitions wherever their limited support permits. The test partition will remain untouched during preprocessing, feature ranking, hyperparameter selection, and knee-point selection.

In [12]:
from sklearn.model_selection import train_test_split

RANDOM_SEEDS = [42, 123, 456, 789, 2024]
PRIMARY_SEED = RANDOM_SEEDS[0]

all_indices = np.arange(len(full_df))
stratification_labels = (
    full_df["diagnostic_label"].astype(str).to_numpy()
)

# First split: 70% training and 30% temporary data
train_indices, temporary_indices = train_test_split(
    all_indices,
    test_size=0.30,
    random_state=PRIMARY_SEED,
    stratify=stratification_labels
)

# Second split: divide temporary data equally into validation and test
validation_indices, test_indices = train_test_split(
    temporary_indices,
    test_size=0.50,
    random_state=PRIMARY_SEED,
    stratify=stratification_labels[temporary_indices]
)

split_names = np.empty(len(full_df), dtype=object)
split_names[train_indices] = "Train"
split_names[validation_indices] = "Validation"
split_names[test_indices] = "Test"

split_distribution_df = pd.crosstab(
    full_df["diagnostic_label"],
    pd.Series(split_names, name="Split")
)

split_distribution_df["Total"] = split_distribution_df.sum(axis=1)
split_distribution_df = split_distribution_df.sort_values(
    "Total",
    ascending=False
)

print("Training rows:", len(train_indices))
print("Validation rows:", len(validation_indices))
print("Test rows:", len(test_indices))
print("Total rows:", (
    len(train_indices) +
    len(validation_indices) +
    len(test_indices)
))

print("\nAttack-label support in every partition:")
display(
    split_distribution_df[
        ["Train", "Validation", "Test", "Total"]
    ]
)

Training rows: 1459304
Validation rows: 312708
Test rows: 312709
Total rows: 2084721

Attack-label support in every partition:


Split,Train,Validation,Test,Total
diagnostic_label,,,,
BENIGN,1107710,237367,237367,1582444
Portscan,111341,23859,23859,159059
DoS Hulk,110928,23770,23770,158468
DDoS,66601,14271,14272,95144
Infiltration - Portscan,48034,10293,10293,68620
DoS GoldenEye,5297,1135,1135,7567
FTP-Patator,2780,596,596,3972
DoS Slowloris,2701,579,579,3859
SSH-Patator,2073,444,444,2961


## Training-Derived Imputation

Infinite values are converted to missing values before modelling. A median imputer is fitted exclusively on the training partition and then applied unchanged to the validation and test partitions. This prevents information from the evaluation data influencing preprocessing. The resulting feature matrices are stored as 32-bit floating-point values to reduce memory use without changing the experimental design. Scaling is not applied here because it will be included only in the Logistic Regression pipeline.

In [13]:
from sklearn.impute import SimpleImputer
import gc

# Create feature matrices for the primary split
X_train_raw = (
    full_df.loc[train_indices, model_feature_columns]
    .replace([np.inf, -np.inf], np.nan)
    .astype(np.float32)
)

X_validation_raw = (
    full_df.loc[validation_indices, model_feature_columns]
    .replace([np.inf, -np.inf], np.nan)
    .astype(np.float32)
)

X_test_raw = (
    full_df.loc[test_indices, model_feature_columns]
    .replace([np.inf, -np.inf], np.nan)
    .astype(np.float32)
)

# Create binary targets
y_train = full_df.loc[
    train_indices, "binary_target"
].to_numpy(dtype=np.int8)

y_validation = full_df.loc[
    validation_indices, "binary_target"
].to_numpy(dtype=np.int8)

y_test = full_df.loc[
    test_indices, "binary_target"
].to_numpy(dtype=np.int8)

# Retain corrected labels for later per-attack test evaluation
diagnostic_labels_test = (
    full_df.loc[test_indices, "diagnostic_label"]
    .astype(str)
    .to_numpy()
)

# Fit preprocessing only on training data
median_imputer = SimpleImputer(strategy="median")

X_train = median_imputer.fit_transform(
    X_train_raw
).astype(np.float32)

X_validation = median_imputer.transform(
    X_validation_raw
).astype(np.float32)

X_test = median_imputer.transform(
    X_test_raw
).astype(np.float32)

feature_names = np.array(model_feature_columns)

# Remove temporary copies
del X_train_raw, X_validation_raw, X_test_raw
gc.collect()

affected_features = quality_audit_df.loc[
    quality_audit_df["Infinite values"] > 0,
    "Feature"
].tolist()

imputation_summary_df = pd.DataFrame({
    "Feature": feature_names,
    "Training median": median_imputer.statistics_
})

imputation_summary_df = imputation_summary_df[
    imputation_summary_df["Feature"].isin(affected_features)
].reset_index(drop=True)

print("Training matrix:", X_train.shape)
print("Validation matrix:", X_validation.shape)
print("Test matrix:", X_test.shape)

remaining_missing = (
    np.isnan(X_train).sum() +
    np.isnan(X_validation).sum() +
    np.isnan(X_test).sum()
)

remaining_infinite = (
    np.isinf(X_train).sum() +
    np.isinf(X_validation).sum() +
    np.isinf(X_test).sum()
)

print("Remaining missing values:", remaining_missing)
print("Remaining infinite values:", remaining_infinite)

print("\nTraining-derived medians for affected features:")
display(imputation_summary_df)

Training matrix: (1459304, 84)
Validation matrix: (312708, 84)
Test matrix: (312709, 84)
Remaining missing values: 0
Remaining infinite values: 0

Training-derived medians for affected features:


,Feature,Training median
0,Flow Bytes/s,3844.659668
1,Flow Packets/s,72.288284


## Mutual Information Feature Ranking

Mutual Information is calculated using only the imputed training partition. Features with no more than 20 distinct training values are treated as discrete indicators; the remaining features are treated as continuous. The ranking is model-independent and does not use validation or test information. Features are ordered from highest to lowest Mutual Information score, with alphabetical ordering used to resolve exact ties reproducibly.

In [14]:
from sklearn.feature_selection import mutual_info_classif
from time import perf_counter
import inspect
import sklearn

# Identify low-cardinality features using training data only
training_unique_counts = np.array([
    np.unique(X_train[:, feature_index]).size
    for feature_index in range(X_train.shape[1])
])

discrete_feature_mask = training_unique_counts <= 20

print("Scikit-learn version:", sklearn.__version__)
print("Features treated as discrete:",
      int(discrete_feature_mask.sum()))
print("Features treated as continuous:",
      int((~discrete_feature_mask).sum()))

# Use all processor cores when supported by the installed version
mi_arguments = {
    "X": X_train,
    "y": y_train,
    "discrete_features": discrete_feature_mask,
    "n_neighbors": 3,
    "random_state": PRIMARY_SEED
}

if "n_jobs" in inspect.signature(
    mutual_info_classif
).parameters:
    mi_arguments["n_jobs"] = -1

start_time = perf_counter()

mutual_information_scores = mutual_info_classif(
    **mi_arguments
)

mi_runtime_seconds = perf_counter() - start_time

mutual_information_df = pd.DataFrame({
    "Feature": feature_names,
    "Mutual Information": mutual_information_scores,
    "Discrete": discrete_feature_mask,
    "Training unique values": training_unique_counts
})

mutual_information_df = (
    mutual_information_df
    .sort_values(
        ["Mutual Information", "Feature"],
        ascending=[False, True],
        kind="mergesort"
    )
    .reset_index(drop=True)
)

mutual_information_df.insert(
    0,
    "Rank",
    np.arange(1, len(mutual_information_df) + 1)
)

ranked_feature_names = mutual_information_df[
    "Feature"
].to_numpy()

print(f"\nMutual Information runtime: "
      f"{mi_runtime_seconds / 60:.2f} minutes")

print("\nTop 20 ranked features:")
display(mutual_information_df.head(20))

Scikit-learn version: 1.6.1
Features treated as discrete: 13
Features treated as continuous: 71

Mutual Information runtime: 16.35 minutes

Top 20 ranked features:


,Rank,Feature,Mutual Information,Discrete,Training unique values
0,1,Bwd Segment Size Avg,0.461539,False,103816
1,2,Bwd Packet Length Mean,0.461465,False,103816
2,3,Total Length of Bwd Packet,0.452316,False,51205
3,4,Total TCP Flow Time,0.449145,False,514748
4,5,Average Packet Size,0.445927,False,151278
5,6,Packet Length Mean,0.445858,False,151278
6,7,Subflow Bwd Bytes,0.443050,False,1555
7,8,Packet Length Std,0.435392,False,270736
8,9,Packet Length Variance,0.435198,False,271371
9,10,Packet Length Max,0.434930,False,5454


## Duplicate Feature-Column Audit

Some CICIDS2017 features may express identical information under different names. Retaining exact duplicate columns would give the same underlying measurement multiple positions in the feature ranking and could distort the progressive reduction analysis. Duplicate-feature decisions are based only on the training partition and are then checked, but not selected, using validation and test data.

In [15]:
import hashlib
from collections import defaultdict

def create_column_signature(values):
    contiguous_values = np.ascontiguousarray(values)
    return hashlib.sha256(
        contiguous_values.tobytes()
    ).hexdigest()

# Generate signatures using training data only
signature_groups = defaultdict(list)

for feature_index, feature_name in enumerate(feature_names):
    signature = create_column_signature(
        X_train[:, feature_index]
    )
    signature_groups[signature].append(feature_name)

duplicate_feature_groups = [
    sorted(group)
    for group in signature_groups.values()
    if len(group) > 1
]

duplicate_feature_groups = sorted(
    duplicate_feature_groups,
    key=lambda group: group[0]
)

verification_records = []

for group_number, group in enumerate(
    duplicate_feature_groups,
    start=1
):
    representative = group[0]
    representative_index = np.where(
        feature_names == representative
    )[0][0]

    for duplicate_feature in group[1:]:
        duplicate_index = np.where(
            feature_names == duplicate_feature
        )[0][0]

        verification_records.append({
            "Group": group_number,
            "Representative": representative,
            "Duplicate feature": duplicate_feature,
            "Identical in training": np.array_equal(
                X_train[:, representative_index],
                X_train[:, duplicate_index]
            ),
            "Identical in validation": np.array_equal(
                X_validation[:, representative_index],
                X_validation[:, duplicate_index]
            ),
            "Identical in test": np.array_equal(
                X_test[:, representative_index],
                X_test[:, duplicate_index]
            )
        })

duplicate_feature_audit_df = pd.DataFrame(
    verification_records
)

print("Duplicate feature groups:",
      len(duplicate_feature_groups))

print("Individual duplicate columns:",
      len(duplicate_feature_audit_df))

display(duplicate_feature_audit_df)

Duplicate feature groups: 3
Individual duplicate columns: 3


,Group,Representative,Duplicate feature,Identical in training,Identical in validation,Identical in test
0,1,Average Packet Size,Packet Length Mean,True,True,True
1,2,Bwd Packet Length Mean,Bwd Segment Size Avg,True,True,True
2,3,Fwd Packet Length Mean,Fwd Segment Size Avg,True,True,True


### Duplicate-Feature Removal

Three features are removed because their values are exactly identical to another feature across the training, validation, and test partitions. The alphabetically first feature in each duplicate group is retained as a deterministic rule. Removing these columns reduces the candidate feature space from 84 to 81 without removing unique information. Mutual Information does not need to be recalculated because it was computed independently for each feature.

In [16]:
# Remove the alphabetically later feature from every duplicate group
duplicate_features_to_drop = [
    group_feature
    for group in duplicate_feature_groups
    for group_feature in group[1:]
]

print("Duplicate features removed:")
for feature in duplicate_features_to_drop:
    print("-", feature)

feature_keep_mask = ~np.isin(
    feature_names,
    duplicate_features_to_drop
)

# Apply the same feature removal to every partition
X_train = X_train[:, feature_keep_mask]
X_validation = X_validation[:, feature_keep_mask]
X_test = X_test[:, feature_keep_mask]

feature_names = feature_names[feature_keep_mask]
training_unique_counts = training_unique_counts[
    feature_keep_mask
]
discrete_feature_mask = discrete_feature_mask[
    feature_keep_mask
]

final_model_feature_columns = feature_names.tolist()

# Remove duplicate columns from the existing univariate MI ranking
mutual_information_df = (
    mutual_information_df[
        ~mutual_information_df["Feature"].isin(
            duplicate_features_to_drop
        )
    ]
    .reset_index(drop=True)
)

mutual_information_df["Rank"] = np.arange(
    1,
    len(mutual_information_df) + 1
)

ranked_feature_names = mutual_information_df[
    "Feature"
].to_numpy()

gc.collect()

print("\nFinal candidate features:", len(feature_names))
print("Training matrix:", X_train.shape)
print("Validation matrix:", X_validation.shape)
print("Test matrix:", X_test.shape)

print("\nFinal top 20 Mutual Information features:")
display(mutual_information_df.head(20))

Duplicate features removed:
- Packet Length Mean
- Bwd Segment Size Avg
- Fwd Segment Size Avg

Final candidate features: 81
Training matrix: (1459304, 81)
Validation matrix: (312708, 81)
Test matrix: (312709, 81)

Final top 20 Mutual Information features:


,Rank,Feature,Mutual Information,Discrete,Training unique values
0,1,Bwd Packet Length Mean,0.461465,False,103816
1,2,Total Length of Bwd Packet,0.452316,False,51205
2,3,Total TCP Flow Time,0.449145,False,514748
3,4,Average Packet Size,0.445927,False,151278
4,5,Subflow Bwd Bytes,0.443050,False,1555
5,6,Packet Length Std,0.435392,False,270736
6,7,Packet Length Variance,0.435198,False,271371
7,8,Packet Length Max,0.434930,False,5454
8,9,Bwd Packet Length Max,0.427383,False,4679
9,10,Flow Duration,0.400132,False,630073
